In [1]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import numpy as np

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=30,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

# Load one food image
image = load_img("transfer_learning_dataset/food_images/sample.jpg",
                 target_size=(224,224))

image = img_to_array(image)
image = np.expand_dims(image, axis=0)

# Generate augmented images
aug_iter = datagen.flow(image, batch_size=1)

plt.figure(figsize=(12,8))

for i in range(5):
    plt.subplot(2,3,i+1)
    batch = next(aug_iter)
    plt.imshow(batch[0].astype("uint8"))
    plt.axis("off")

plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'transfer_learning_dataset/food_images/sample.jpg'

In [ ]:
import numpy as np
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load VGG16
base_model = VGG16(
    weights='imagenet',
    include_top=False
)

generator = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

data = generator.flow_from_directory(
    "transfer_learning_dataset/sneaker_images",
    target_size=(224,224),
    batch_size=30,
    class_mode=None,
    shuffle=False
)

features = base_model.predict(data)

print("Feature Shape:", features.shape)

np.save("vgg16_sneaker_features.npy", features)

print("Features saved successfully.")

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze all layers
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze last 20 layers
for layer in base_model.layers[-20:]:
    layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

train_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    "transfer_learning_dataset/glasses_dataset/train",
    target_size=(224,224),
    batch_size=8,
    class_mode="binary"
)

val_data = train_gen.flow_from_directory(
    "transfer_learning_dataset/glasses_dataset/validation",
    target_size=(224,224),
    batch_size=8,
    class_mode="binary"
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
print("Comparison Between Feature Extraction and Fine-Tuning\n")

print("Feature Extraction:")
print("- All pretrained layers remain frozen.")
print("- Faster training.")
print("- Lower risk of overfitting.")
print("- Works well on small datasets.\n")

print("Fine-Tuning:")
print("- Last layers are trainable.")
print("- Higher validation accuracy on similar datasets.")
print("- Greater risk of overfitting.")
print("- Requires more training time.\n")

print("Summary:")
print("For a small T-shirt dataset, feature extraction generally provides better generalization because it keeps the pretrained features unchanged and reduces overfitting. Fine-tuning can achieve higher accuracy when enough training data is available but may overfit if the dataset is too small.")

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
train_data = ImageDataGenerator(rescale=1./255).flow_from_directory(
    "transfer_learning_dataset/headphone_dataset/train",
    target_size=(224,224),
    batch_size=8,
    class_mode="categorical"
)

validation_data = ImageDataGenerator(rescale=1./255).flow_from_directory(
    "transfer_learning_dataset/headphone_dataset/validation",
    target_size=(224,224),
    batch_size=8,
    class_mode="categorical"
)

history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=5
)

print("Training completed successfully.")